For licensing see accompanying LICENSE file.  
Copyright (C) 2025 Apple Inc. All Rights Reserved.

# Cost Analysis of Feature Description Methods
Relies on methods outputting their call responses. To compute, uncomment the print lines in each feature description method.

In [4]:
%load_ext autoreload
%autoreload 2

In [ ]:
import io
import os
import sys
import numpy as np
from tqdm import tqdm
from pathlib import Path
from contextlib import redirect_stdout

ROOT = Path.cwd().parent
os.chdir(ROOT) # Change working directory to project root
sys.path.insert(0, str(ROOT))

import methods
import features


In [29]:
def compute_cost_analysis(method, num_layers, num_indices_per_layer, feature_model_name, feature_source_name, exp_eval_model_name):
    num_completion_tokens = []
    num_prompt_tokens = []
    for layer in tqdm(range(num_layers)):
        for index in range(num_indices_per_layer):
            feature = features.Feature(feature_model_name, f'{layer}-{feature_source_name}', index)
            try:
                output = io.StringIO()
                with redirect_stdout(output):
                    method.generate(feature, exp_eval_model_name)
                cost = output.getvalue().split('COST')[1].strip()
                n_completion, n_prompt = [int(n) for n in cost.split()]
                num_completion_tokens.append(n_completion)
                num_prompt_tokens.append(n_prompt)
            except Exception as e:
                continue

    print(f"Avg completion tokens: {np.mean(num_completion_tokens)}")
    print(f"Avg prompt tokens: {np.mean(num_prompt_tokens)}")
    return num_completion_tokens, num_prompt_tokens

In [30]:
gpt_num_layers = 13
gpt_num_indices_per_layer = 10
gpt_model_name = 'gpt2-small'
gpt_feature_source = 'res-jb'
exp_eval_model = 'gpt-4o-mini'

In [31]:
feature_description_methods = [
    methods.OAITokenActPair('./', 0, None, None),
    methods.EleutherActsTop20('./', 0, None, None),
    methods.SemanticRegex('./', 0, None, None)
]
results = {}
for method in feature_description_methods:
    print(f"Method: {method.name}")
    n_completion_tokens, n_prompt_tokens = compute_cost_analysis(
        method,
        gpt_num_layers,
        gpt_num_indices_per_layer,
        gpt_model_name,
        gpt_feature_source,
        exp_eval_model
    )
    results[method.name] = (n_completion_tokens, n_prompt_tokens)

Method: oai_token-act-pair


100%|██████████| 13/13 [02:03<00:00,  9.47s/it]


Avg completion tokens: 8.564516129032258
Avg prompt tokens: 1375.766129032258
Method: eleuther_acts_top20


100%|██████████| 13/13 [02:45<00:00, 12.77s/it]


Avg completion tokens: 30.35483870967742
Avg prompt tokens: 1007.2016129032259
Method: semantic_regex


100%|██████████| 13/13 [03:19<00:00, 15.33s/it]

Avg completion tokens: 33.71774193548387
Avg prompt tokens: 1230.25


In [ ]:
oai_system_prompt_tokens = 919
eleuther_system_prompt_tokens = 483
semantic_system_regex_prompt_tokens = 993
system_prompt_tokens = {
    "oai_token-act-pair": oai_system_prompt_tokens,
    "eleuther_acts_top20": eleuther_system_prompt_tokens,
    "semantic_regex": semantic_system_regex_prompt_tokens
}

# Reruning this cell without commenting out the example message in the feature description methods will not work.
for method in feature_description_methods:
    feature = features.Feature(gpt_model_name, f'0-{gpt_feature_source}', 0)
    try:
        output = io.StringIO()
        with redirect_stdout(output):
            method.generate(feature, exp_eval_model)
        cost = output.getvalue().split('COST')[1].strip()
        _, n_prompt = [int(n) for n in cost.split()]
        print(f"{method.name} system and few-shot tokens: {n_prompt}")
    except Exception as e:
        continue


oai_token-act-pair system and few-shot tokens: 919
eleuther_acts_top20 system and few-shot tokens: 483
semantic_regex system and few-shot tokens: 993


In [39]:
for result_name, result in results.items():
    n_completion_tokens, n_prompt_tokens = result
    n_prompt_tokens = [n - system_prompt_tokens[result_name] for n in n_prompt_tokens]
    print(f"Method: {result_name}")
    print(f"Avg completion tokens: {np.mean(n_completion_tokens)} +/- {np.std(n_completion_tokens)}")
    print(f"Avg prompt tokens: {np.mean(n_prompt_tokens)} +/- {np.std(n_prompt_tokens)}")
    print()

Method: oai_token-act-pair
Avg completion tokens: 8.564516129032258 +/- 2.508829361648354
Avg prompt tokens: 456.76612903225805 +/- 212.91328345622867

Method: eleuther_acts_top20
Avg completion tokens: 30.35483870967742 +/- 6.912113663458238
Avg prompt tokens: 524.2016129032259 +/- 89.64742587013015

Method: semantic_regex
Avg completion tokens: 33.71774193548387 +/- 11.823678050550942
Avg prompt tokens: 237.25 +/- 46.55496664540571

